# Fleet Delivery Fuel Consumption
Regression case study | 60 marks | 20 tests

Run the definition cells first. Run training and prediction separately from this folder. Prediction requires the saved model, but does not require training again.

In [ ]:
import numpy as np
import pandas as pd
import joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RAW_NUMERIC = ["distance_km", "trip_duration_hours", "payload_kg",
               "ambient_temperature_c", "idle_minutes"]
numeric_cols = RAW_NUMERIC
numeric_features = numeric_cols + ["start_hour", "average_speed_kmph"]
category_cols = ["depot_code", "vehicle_type", "route_type"]
features = numeric_features + category_cols
TARGET = "fuel_consumed_litres"

### load_data

In [ ]:
def load_data(filename: str) -> pd.DataFrame:
    """Return a cleaned DataFrame with the original data and engineered features.

    Return type: pandas.DataFrame. The same function accepts train, test, or
    prediction data; it does not need a training flag.
    """

    df = pd.read_csv(filename,dtype={'trip_id':'string'})
    df = df.drop_duplicates(subset=['trip_id'],keep='first')

    df['trip_start_ts'] = pd.to_datetime(df['trip_start_ts'],errors='coerce',format='mixed',dayfirst=True)
    df = df.dropna(subset=['trip_start_ts'])

    for col in RAW_NUMERIC:
        df[col] = pd.to_numeric(df[col],errors='coerce').replace([np.inf,-np.inf],np.nan)

    for col in category_cols:
        df[col] = df[col].astype('string').str.strip().str.lower().replace('',pd.NA).fillna('unknown')

    df['start_hour'] = df['trip_start_ts'].dt.hour
    df['average_speed_kmph'] = (df['distance_km'] / df['trip_duration_hours'].where(df['trip_duration_hours']>0)).replace([np.inf,-np.inf],np.nan)
    
    return df
    raise NotImplementedError('Complete load_data')


### build_pipeline

In [ ]:
def build_pipeline(algorithm) -> Pipeline:
    """Return an UNFITTED sklearn Pipeline containing preprocessing and model.

    Return type: sklearn.pipeline.Pipeline.
    """

    num_pipeline = Pipeline([
        ('imputer',SimpleImputer(strategy='median')),
        ('scaler',StandardScaler())
    ])

    preprocessor = ColumnTransformer([
        ('numeric',num_pipeline,numeric_features),
        ('category',OneHotEncoder(handle_unknown='ignore'),category_cols)
    ])

    return Pipeline([
        ('preprocessor',preprocessor),('model',algorithm)
    ])
    raise NotImplementedError('Complete build_pipeline')


### prepare_datasets

In [ ]:
def prepare_datasets(train_file: str, test_file: str) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    """Return clean predictor/target pairs for train and test CSVs."""

    train_df = load_data(train_file)
    test_df = load_data(test_file)

    for df in train_df,test_df:
        df[TARGET] = pd.to_numeric(df[TARGET],errors='coerce').replace([np.inf,-np.inf],np.nan)
        df.dropna(subset=[TARGET],inplace=True)

    X_train = train_df[features]
    X_test = test_df[features]
    y_train = train_df[TARGET]
    y_test = test_df[TARGET]

    return X_train,X_test,y_train,y_test
    raise NotImplementedError('Complete prepare_datasets')


### train_models

In [ ]:
def train_models(X_train: pd.DataFrame, y_train: pd.Series) -> dict[str, Pipeline]:
    """Return dict[str, fitted sklearn.pipeline.Pipeline]."""

    models = {
        'linear_regression' : LinearRegression(),
        'random_forest' : RandomForestRegressor(n_estimators=150, max_depth=10, min_samples_leaf=2,random_state=42, n_jobs=-1)
    }

    trained_models ={}

    for name,model in models.items():
        pipeline = build_pipeline(model)
        pipeline.fit(X_train,y_train)
        trained_models[name] = pipeline\

    return trained_models
    raise NotImplementedError('Complete train_models')


### evaluate_models

In [ ]:
def evaluate_models(trained_models: dict[str, Pipeline],
                    X_test: pd.DataFrame,
                    y_test: pd.Series) -> pd.DataFrame:
    """Return comparison DataFrame: model_name, mae, rmse, r2; RMSE ascending."""

    results = []

    for name,model in trained_models.items():
        predictions = np.maximum(model.predict(X_test),0)
        results.append({
            'model_name' : name,
            'mae' : mean_absolute_error(y_test,predictions),
            'rmse' : np.sqrt(mean_squared_error(y_test,predictions)),
            'r2' : r2_score(y_test,predictions)
        })

    comparison = pd.DataFrame(results)

    return comparison.sort_values(['rmse']).reset_index(drop=True)
    raise NotImplementedError('Complete evaluate_models')


### Function 6 — Select the best model

In [ ]:
def select_best_model(comparison: pd.DataFrame, trained_models: dict[str, Pipeline]) -> tuple[str, Pipeline]:
    """Return the first-ranked model name and its existing fitted pipeline."""

    best_name = comparison.iloc[0]['model_name']
    best_model = trained_models[best_name]

    return best_name,best_model
    raise NotImplementedError('Complete select_best_model')


### predict_fuel

In [ ]:
def predict_fuel(model: Pipeline, df: pd.DataFrame) -> pd.DataFrame:
    """Return DataFrame[trip_id, predicted_fuel_consumed_litres].

    Clip negative estimates to zero, round to one decimal and preserve order.
    """

    predictions = np.round(np.maximum(model.predict(df[features]),0),1)
    
    return pd.DataFrame({
        'trip_id' : df['trip_id'].to_numpy(),
        'predicted_fuel_consumed_litres' : predictions
    })
    raise NotImplementedError('Complete predict_fuel')


### train

In [ ]:
def train() -> tuple[str, pd.DataFrame]:
    """Fit, compare, persist; return (best_name: str, comparison: DataFrame)."""

    X_train,X_test,y_train,y_test = prepare_datasets('fleet_train.csv','fleet_test.csv')

    trained_models = train_models(X_train,y_train)

    comparison = evaluate_models(trained_models,X_test,y_test)

    best_name , best_model = select_best_model(comparison,trained_models)

    joblib.dump(best_model,'best_model.pkl')
    comparison.to_csv('model_comparison.csv',index=False)
    
    return best_name,comparison
    raise NotImplementedError('Complete train')


### predict

In [ ]:
def predict() -> pd.DataFrame:
    """Load a fitted model and return the queued prediction DataFrame."""

    data = load_data('fleet_predict.csv')
    model = joblib.load('best_model.pkl')

    predictions = predict_fuel(model,data)

    predictions.to_csv('fleet_predictions.csv',index=False)

    return predictions
    raise NotImplementedError('Complete predict')


### Training workflow
Execute only when training or refreshing the model.

In [ ]:
# Uncomment to train after completing the functions:
# best_name, comparison = train()
# display(comparison)


### Prediction workflow
Run the definitions and this cell in a fresh session after training has produced best_model.pkl.

In [ ]:
# Uncomment to predict after training:
# predictions = predict()
# display(predictions.head())
